Wczytanie odpowiednich bibliotek

In [1]:
import polars as pl
import matplotlib.pyplot as plt
import numpy as np
import os
from matplotlib import colormaps

Zmiana katalogu roboczego

In [2]:
os.chdir('../..')

Wczytanie danych wynikowych

In [4]:
pl_data = pl.read_parquet('experiments/output/experiments_data.parquet')
print(f'Jest {pl_data.shape[0]:,} obserwacji i {pl_data.shape[1]:,} kolumn')

Jest 2,908,800 obserwacji i 35 kolumn


In [5]:
print(pl_data.head)

<bound method DataFrame.head of shape: (2_908_800, 35)
┌────────────┬─────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ dataset    ┆ cpu_gpu ┆ balanced  ┆ model_con ┆ … ┆ model_spa ┆ ignored_v ┆ train_pre ┆ test_pred │
│ ---        ┆ ---     ┆ ---       ┆ fig_num   ┆   ┆ rsity     ┆ ariables_ ┆ d_time    ┆ _time     │
│ str        ┆ str     ┆ str       ┆ ---       ┆   ┆ ---       ┆ count     ┆ ---       ┆ ---       │
│            ┆         ┆           ┆ i64       ┆   ┆ f64       ┆ ---       ┆ f64       ┆ f64       │
│            ┆         ┆           ┆           ┆   ┆           ┆ i64       ┆           ┆           │
╞════════════╪═════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪═══════════╡
│ japanese   ┆ cpu     ┆ balanced  ┆ 8         ┆ … ┆ 0.68      ┆ 1         ┆ 0.052602  ┆ 0.045218  │
│ japanese   ┆ cpu     ┆ balanced  ┆ 8         ┆ … ┆ 0.68      ┆ 1         ┆ 0.048895  ┆ 0.046869  │
│ japanese   ┆ cpu     ┆ balanced  ┆

In [ ]:
pl_data.pivot('cpu_gpu', index='dataset', aggregate_function='sum')

Sprawdzenie, czy GPU lending club dla modelu nr 0 jest uwzględniony


In [21]:
pl_data1 = pl_data.filter( 
            pl.col('cpu_gpu')=='gpu', 
            pl.col('model_config_num') == 0,
            pl.col('fold') == 1).select('dataset').unique()
print(pl_data1)

shape: (2, 1)
┌─────────────────────┐
│ dataset             │
│ ---                 │
│ str                 │
╞═════════════════════╡
│ japanese            │
│ give-me-some-credit │
└─────────────────────┘


In [19]:
pl_data.filter(pl.col('cpu_gpu')=='gpu').group_by(['model_config_num','balanced','dataset']).len().sort(['model_config_num','balanced','dataset']).filter(pl.col("len") != 3600) 

model_config_num,balanced,dataset,len
i64,str,str,u32
0,"""balanced""","""give-me-some-credit""",1200
0,"""unbalanced""","""give-me-some-credit""",1200


Wybór danych dla jednego datastu i jednego foldu

In [ ]:
pl_data1 = pl_data.filter(pl.col('balanced')=='balanced', 
            pl.col('cpu_gpu')=='cpu', 
            pl.col('dataset') == 'give-me-some-credit',
            pl.col('fold') == 1)

In [ ]:
from src.experiments.analyze_dataset import show_scatter_plot_epochs

In [ ]:
show_scatter_plot_epochs(pl_data1, 'val_precision', 'val_recall')

In [ ]:
from itertools import combinations 
lst_metrics = ['val_auc',
'val_binary_accuracy',
'val_loss',
'val_precision',
'val_recall',
'train_time',
'validation_time',
'train_pred_time',
'test_pred_time']
tup_combinations  = tuple(combinations(lst_metrics, 2))

In [ ]:
for metric_x, metric_y in tup_combinations:
    # print(metric_x, metric_y)
    show_scatter_plot_epochs(pl_data1, metric_x, metric_y)

In [ ]:
pl_data1 = pl_data.filter(pl.col('balanced')=='balanced', 
            pl.col('cpu_gpu')=='cpu', 
            pl.col('dataset') == 'german-credit-data',
            pl.col('fold') == 1)

In [ ]:
for metric_x, metric_y in tup_combinations:
    # print(metric_x, metric_y)
    show_scatter_plot_epochs(pl_data1, metric_x, metric_y)